# 🛠️ Tool 1 — NumPy : Fondations, Vectorisation et Masques Booléens

Bienvenue dans le premier module d'outils du cours **Data Science & Python** !

Avant d'aborder les analyses approfondies (comme dans les projets *Energy Lab* et la détection d'appareils électriques), il est indispensable de maîtriser notre boîte à outils fondamentale.

Au cœur de quasiment toutes les bibliothèques modernes de Data Science et d'IA en Python (**Pandas**, **Scikit-Learn**, **PyTorch**, **SciPy**, **Matplotlib**), se trouve le moteur universel du calcul numérique : **NumPy** (*Numerical Python*).

---

### 🎯 Objectifs de ce notebook :
1. **Comprendre l'architecture** : pourquoi les listes Python sont inadaptées aux gros volumes de données et comment fonctionne un `np.ndarray` en mémoire (bloc continu, type homogène).
2. **Créer et inspecter** des tableaux 1D (`array`, `zeros`, `ones`, `arange`, `linspace`, attributs `shape`, `dtype`, `size`).
3. **Maîtriser la vectorisation** et éliminer définitivement les boucles `for` lentes.
4. **Éviter le piège n°1 de NumPy** : la différence fondamentale entre une **Vue** (`view`) et une **Copie** (`copy`).
5. **Filtrer des données** grâce aux masques booléens (`&`, `|`, `~`) et à la fonction conditionnelle `np.where`.
6. **Grand Cas Pratique Réel** : auditer et analyser **99 396 mesures réelles** de consommation électrique d'une maison (relevé Linky de près de deux semaines) sans la moindre boucle !

> 💡 **Pédagogie interactive** : Ce notebook contient des **QCM interactifs** et des cellules de vérification automatique `check()` pour valider vos réponses en direct.  
> 👉 `Shift + Entrée` pour exécuter chaque cellule pas à pas.

In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Relances humoristiques adaptées à l'univers NumPy & Data Science
_RELANCES_HUMOUR = [
    "Une boucle for en pur Python va plus vite pour trouver la bonne réponse !",
    "Erreur de dimension (BroadcastError) dans ton raisonnement !",
    "Aïe... tu as confondu une copie et une vue en mémoire !",
    "Pas d'inquiétude, même Guido van Rossum a dû relire la doc de NumPy un jour.",
    "Attention au type : ton tableau attendait un entier et a reçu un float tronqué !",
    "Même le garbage collector de Python n'aurait pas osé cette réponse. Retente !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-5):
    """
    Vérifie la valeur d'une variable ou d'un array calculé dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and np.allclose(valeur, attendu, atol=tol)
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(valeur - attendu) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(abs(v - a) <= tol for v, a in zip(valeur, attendu))
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement initialisé avec NumPy version {np.__version__}")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")

---
## 1 · Pourquoi NumPy ? Le problème des listes Python

En Python standard, la structure universelle pour stocker une suite de valeurs est la **liste** (`list`).  
Pourtant, dès qu'on manipule des milliers ou des millions de mesures, les listes deviennent très lentes et consomment énormément de mémoire.

### 🧠 En mémoire : Pointers vs Mémoire Contiguë
- **Une liste Python** est un tableau de **pointeurs** vers des objets dispersés n'importe où dans la RAM. Chaque entier Python (`PyObject`) pèse environ 28 octets à cause des métadonnées (type, compteur de références).
- **Un array NumPy (`np.ndarray`)** est un bloc de mémoire **contigu** (comme en langage C). Un entier 64 bits n'occupe exactement que 8 octets, sans aucun emballage d'objet !

```text
Structure d'une Liste Python en mémoire :
[ Pointeur 1 ] ───► [ PyObject int : 12 ] (28 octets éparpillés dans la RAM)
[ Pointeur 2 ] ───► [ PyObject int : 15 ] (28 octets éparpillés dans la RAM)
[ Pointeur 3 ] ───► [ PyObject int : 22 ] (28 octets éparpillés dans la RAM)
➡️ Cache CPU inefficace, parcours par déréférencement de pointeurs.

Structure d'un Array NumPy (int64) en mémoire :
[  12  ][  15  ][  22  ][  18  ][  30  ]
▲       ▲       ▲
└───────┴───────┴──► Bloc contigu de mémoire brute (8 octets par valeur)
➡️ Chargement direct dans les caches L1/L2 du CPU, instructions SIMD vectorisées !
```

Voyons la différence de vitesse en temps réel sur la somme de **2 millions de nombres** :

In [6]:
# Programme 2 — Benchmark : Liste Python vs NumPy Array
import time

N = 2_000_000

# 1. Somme avec une liste Python pure
liste_python = list(range(N))
t0 = time.perf_counter()
somme_py = sum(liste_python)
duree_py = time.perf_counter() - t0

# 2. Somme avec un array NumPy
array_numpy = np.arange(N, dtype=np.int64)
t0 = time.perf_counter()
somme_np = np.sum(array_numpy)
duree_np = time.perf_counter() - t0

speedup = duree_py / duree_np
print(f"Somme Python (list + sum) : {duree_py*1000:6.1f} ms")
print(f"Somme NumPy (arange + sum): {duree_np*1000:6.1f} ms")
print(f"🚀 NumPy est environ {speedup:.1f}x plus rapide !")

Somme Python (list + sum) :   44.8 ms
Somme NumPy (arange + sum):    0.4 ms
🚀 NumPy est environ 110.8x plus rapide !


In [7]:
# Programme 3 — QCM 1
qcm(
    "Pourquoi un array NumPy est-il beaucoup plus rapide et économe en mémoire qu'une liste Python ?",
    [
        "Parce que NumPy compresse les données en format ZIP dans la RAM.",
        "Parce que les données sont stockées de façon contiguë en mémoire et exécutées en code C optimisé.",
        "Parce que NumPy ignore les nombres après la virgule pour aller plus vite.",
        "Parce que Python limite volontairement la vitesse de ses listes pour éviter la surchauffe du processeur."
    ],
    correct=1,
    explication="NumPy alloue un bloc continu de mémoire brute typée. Le processeur peut ainsi charger les données par blocs entiers dans ses registres SIMD ultra-rapides sans déréférencer de pointeurs d'objets Python."
)

---
## 2 · Création et anatomie d'un array NumPy

Pour créer un tableau NumPy, on peut partir d'une liste existante ou utiliser des constructeurs natifs.

### 2.1 Les constructeurs fréquents
| Fonction | Description | Exemple |
|---|---|---|
| `np.array(liste)` | Convertit une liste Python en array | `np.array([10, 20, 30])` |
| `np.zeros(shape)` | Crée un array rempli de zéros | `np.zeros(5)` |
| `np.ones(shape)` | Crée un array rempli de uns | `np.ones(5)` |
| `np.full(shape, val)` | Remplit un array avec une valeur constante | `np.full(4, 50.0)` |
| `np.arange(start, stop, step)` | Suite de nombres avec un **pas** régulier | `np.arange(0, 10, 2)` → `[0, 2, 4, 6, 8]` |
| `np.linspace(start, stop, num)` | Suite de `num` points régulièrement espacés | `np.linspace(0, 1, 5)` → `[0, 0.25, 0.5, 0.75, 1]` |

> ⚠️ **Différence capitale : `arange` vs `linspace`**
> - Avec `np.arange`, on choisit le **pas** (la borne finale est exclue, attention aux arrondis sur les flottants).
> - Avec `np.linspace`, on choisit le **nombre exact de points** (la borne finale est **incluse** par défaut). En traitement du signal et séries temporelles, on utilise quasi systématiquement `linspace` pour construire les axes de temps.

### 2.2 Les 4 attributs vitaux à inspecter
- `.ndim` : nombre de dimensions (1 pour un vecteur 1D).
- `.shape` : la forme sous forme de tuple (ex: `(251,)`).
- `.size` : nombre total d'éléments.
- `.dtype` : le type des éléments (`int32`, `int64`, `float64`, `bool_`, etc.). Tous les éléments d'un array ont **obligatoirement le même type** !

In [4]:
# Programme 4 — Création et inspection d'un array
# Création d'un vecteur de mesures de tension
tensions = np.array([230.1, 230.4, 229.8, 231.0, 229.5])

print("Array tensions :", tensions)
print(f"Nombre de dimensions (.ndim) : {tensions.ndim}")
print(f"Forme du tableau     (.shape): {tensions.shape}")
print(f"Nombre d'éléments    (.size) : {tensions.size}")
print(f"Type des éléments    (.dtype): {tensions.dtype}")

Array tensions : [230.1 230.4 229.8 231.  229.5]
Nombre de dimensions (.ndim) : 1
Forme du tableau     (.shape): (5,)
Nombre d'éléments    (.size) : 5
Type des éléments    (.dtype): float64


In [8]:
# Programme 5 — QCM 2
qcm(
    "Si l'on exécute a = np.array([1, 2, 3.5]), quel sera le dtype du tableau résultant ?",
    [
        "Une erreur est levée car on mélange des entiers et un float.",
        "int64 (les nombres après la virgule sont tronqués).",
        "float64 (les entiers sont convertis en 1.0 et 2.0 pour maintenir l'homogénéité).",
        "object (chaque élément conserve son type d'origine)."
    ],
    correct=2,
    explication="NumPy impose un type homogène pour garantir les performances mémoire. Il effectue un 'upcasting' automatique vers le type le plus général capable de représenter toutes les valeurs sans perte, ici float64."
)

### ✏️ Exercice 1 — À toi de jouer !

Dans le cadre de l'échantillonnage d'un capteur mesurant à **50 Hz** pendant **5 secondes**, créez :
1. Un vecteur `temps` débutant à `0.0` et se terminant à `5.0` contenant exactement **251 points** régulièrement espacés (utilisez `np.linspace`).
2. Calculez sa taille `taille_temps` en utilisant l'attribut adéquat de l'array.

Remplacez les `...` ci-dessous puis exécutez la cellule de test.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
temps = np.linspace(0.0, 5.0, 251)
taille_temps = temps.size
```
</details>

In [ ]:
# Programme 6 — Exercice 1
# Remplacez les '...' par votre code :
temps = ...
taille_temps = ...

In [ ]:
# Programme 7 — Test Exercice 1
# Exécutez cette cellule pour valider votre réponse
check("temps", temps, np.linspace(0.0, 5.0, 251))
check("taille_temps", taille_temps, 251)

---
## 3 · La Vectorisation : Adieu les boucles `for`

En Python pur, si l'on veut ajouter une taxe de 20% à une liste de prix, on est forcé d'écrire une boucle :
```python
prix_ttc = [p * 1.20 for p in prix_ht]
```

Avec NumPy, les opérations s'appliquent directement et instantanément à **tous les éléments** en langage machine :
```python
prix_ttc = prix_ht * 1.20
```

C'est ce qu'on appelle la **vectorisation** (*element-wise operations*).

### 3.1 Opérations arithmétiques et fonctions universelles (`ufuncs`)
- Arithmétique scalaire : `+`, `-`, `*`, `/`, `//`, `**`.
- Fonctions mathématiques universelles : `np.sin()`, `np.cos()`, `np.exp()`, `np.log()`, `np.sqrt()`, `np.abs()`.
- Opérations entre deux arrays de même taille :
  - `a + b` : somme terme à terme.
  - `a * b` : produit terme à terme (**produit de Hadamard**, pas produit matriciel !).

In [11]:
# Programme 8 — Démonstration de vectorisation
t = np.linspace(0, 2 * np.pi, 5)
signal_sin = np.sin(t)

print("t          :", np.round(t, 2))
print("sin(t)     :", np.round(signal_sin, 2))
print("sin(t) * 10:", np.round(signal_sin * 10, 2))

t          : [0.   1.57 3.14 4.71 6.28]
sin(t)     : [ 0.  1.  0. -1. -0.]
sin(t) * 10: [  0.  10.   0. -10.  -0.]


In [12]:
# Programme 9 — QCM 3
qcm(
    "Si a = np.array([2, 3]) et b = np.array([4, 5]), que retourne a * b ?",
    [
        "array([8, 15]) — chaque élément est multiplié par son homologue.",
        "23 — c'est le produit scalaire (2*4 + 3*5).",
        "Une erreur : il faut obligatoirement utiliser une boucle for.",
        "array([[8, 10], [12, 15]]) — le produit tensoriel."
    ],
    correct=0,
    explication="L'opérateur standard * entre deux arrays NumPy applique TOUJOURS la multiplication élément par élément. Pour un produit scalaire ou matriciel, on utilise l'opérateur @ ou la fonction np.dot(a, b)."
)

### ✏️ Exercice 2 — Calcul de puissance sans boucle

On dispose de 5 mesures synchronisées de tension $U$ (en Volts) et d'intensité $I$ (en Ampères) prélevées sur un appareil électrique :
- $U = [230.2, 229.8, 231.1, 230.0, 228.9]$
- $I = [0.65, 0.64, 0.68, 0.65, 0.63]$

Calculez la puissance active instantanée $P = U 	imes I$ (en Watts) en **une seule ligne**, sous la forme d'un array NumPy `puissance`.  
Calculez également la puissance moyenne `puissance_moyenne` avec `.mean()`.

In [ ]:
# Programme 10 — Exercice 2
tension = np.array([230.2, 229.8, 231.1, 230.0, 228.9])
intensite = np.array([0.65, 0.64, 0.68, 0.65, 0.63])

# Remplacez les '...' par votre code vectorisé :
puissance = ...
puissance_moyenne = ...

In [ ]:
# Programme 11 — Test Exercice 2
check("puissance", puissance, tension * intensite)
check("puissance_moyenne", puissance_moyenne, float((tension * intensite).mean()))

---
## 4 · Indexation, Slicing et le Piège Ultime : Vue vs Copie

L'accès aux éléments d'un array 1D ressemble à celui d'une liste Python :
- `arr[0]` : premier élément
- `arr[-1]` : dernier élément
- `arr[2:5]` : tranche des indices 2, 3 et 4
- `arr[::2]` : un élément sur deux

Mais attention ! Il existe une différence fondamentale qui piège même des développeurs expérimentés :

> ⚠️ **LE PIÈGE N°1 DE NUMPY : Un slice crée une VUE (View), PAS une COPIE !**
> - En Python pur, `sous_liste = ma_liste[0:3]` alloue une nouvelle liste indépendante.
> - Dans NumPy, `sous_array = mon_array[0:3]` pointe vers **exactement la même zone mémoire**.
> - Si vous modifiez `sous_array`, vous **modifiez silencieusement `mon_array`** !
>
> 💡 *Pourquoi ce choix ?* Pour des raisons de performance ! Si votre array pèse 4 Go, extraire une tranche ne doit pas dupliquer 4 Go de données en RAM.  
> 👉 Pour obtenir une copie réellement indépendante, il faut appeler explicitement `.copy()`.

In [16]:
# Programme 12 — Démonstration du piège Vue vs Copie
original = np.array([10, 20, 30, 40, 50])
print("Original au départ          :", original)

# Création d'un slice sans .copy()
vue = original[0:2]
vue[0] = 999  # On modifie la vue !

print("Après modification de vue[0]=999 :")
print("  -> vue      :", vue)
print("  -> original :", original, " ⚠️ L'original a été modifié !")

# Maintenant avec une vraie copie indépendante :
original_sain = np.array([10, 20, 30, 40, 50])
vraie_copie = original_sain[0:2].copy()
vraie_copie[0] = 999

print("\nAvec .copy() :")
print("  -> vraie_copie  :", vraie_copie)
print("  -> original_sain:", original_sain, " ✅ L'original est intact !")

Original au départ          : [10 20 30 40 50]
Après modification de vue[0]=999 :
  -> vue      : [999  20]
  -> original : [999  20  30  40  50]  ⚠️ L'original a été modifié !

Avec .copy() :
  -> vraie_copie  : [999  20]
  -> original_sain: [10 20 30 40 50]  ✅ L'original est intact !


In [17]:
# Programme 13 — QCM 4
qcm(
    "Que se passe-t-il si vous écrivez b = a[2:6] puis b[:] = 0 ?",
    [
        "Seul l'array b est mis à 0, a reste inchangé.",
        "Les éléments d'indice 2 à 5 de l'array original a sont également mis à 0, car b est une vue.",
        "NumPy lève une exception PermissionError car un slice est en lecture seule.",
        "Le tableau a est entièrement supprimé de la mémoire."
    ],
    correct=1,
    explication="Le slicing dans NumPy ne duplique pas les données : b est une vue (view) partageant le même buffer mémoire que a. Toute modification de b altère directement a."
)

### ✏️ Exercice 3 — Extraire une copie indépendante

Soit l'array `signal_brut` ci-dessous.
1. Extrayez les 3 derniers éléments de `signal_brut` dans une variable `fin_signal` sous forme d'une **copie indépendante** (utilisez `.copy()`).
2. Modifiez le premier élément de `fin_signal` pour lui assigner la valeur `0.0`.
3. Vérifiez que `signal_brut` n'a pas été altéré.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
fin_signal = signal_brut[-3:].copy()
fin_signal[0] = 0.0
```
</details>

In [ ]:
# Programme 14 — Exercice 3
signal_brut = np.array([12.5, 14.2, 18.9, 21.0, 19.5, 15.0])

# 1. Extrayez les 3 derniers éléments avec .copy()
fin_signal = ...

# 2. Modifiez le 1er élément de fin_signal à 0.0
# fin_signal[0] = 0.0 (à décommenter après avoir créé fin_signal)


In [ ]:
# Programme 15 — Test Exercice 3
check("fin_signal", fin_signal, np.array([0.0, 19.5, 15.0]))
check("signal_brut reste intact", signal_brut, np.array([12.5, 14.2, 18.9, 21.0, 19.5, 15.0]))

---
## 5 · Masquage Booléen et Filtrage (Boolean Indexing)

Le filtrage de données est l'un des piliers de la Data Science : isoler les consommations anormales, éliminer les valeurs négatives, détecter les surchauffes.

### 5.1 Les comparaisons vectorisées créent des masques
Si on compare un array à une valeur, NumPy applique la comparaison à chaque élément et renvoie un **tableau de booléens** :
```python
puissances = np.array([30, 145, 28, 155, 32])
masque = puissances > 100
# masque vaut : array([False,  True, False,  True, False])
```

### 5.2 L'indexation par masque extrait les données
En passant ce masque entre crochets `puissances[masque]`, NumPy ne garde que les éléments où la valeur est `True` :
```python
compresseur_on = puissances[puissances > 100]
# compresseur_on vaut : array([145, 155])
```

### 5.3 Combiner des conditions : `&` (ET), `|` (OU), `~` (NON)
> ⚠️ **Attention aux règles de syntaxe en NumPy :**
> - On **DOIT** utiliser les symboles bitwise : `&` pour ET, `|` pour OU, `~` pour NON.
> - On **NE DOIT PAS** utiliser les mots-clés Python `and` ou `or` (ils provoquent un crash `ValueError: The truth value of an array is ambiguous`).
> - Chaque condition doit obligatoirement être **entourée de parenthèses** à cause de la priorité des opérateurs :
>   `selection = (puissances >= 30) & (puissances <= 150)`

### 5.4 La fonction magique `np.where(condition, valeur_si_vrai, valeur_si_faux)`
C'est le `SI()` d'Excel, mais vectorisé à la vitesse du C.  
Exemple : binariser un état :
```python
etat = np.where(puissances > 50, "MARCHE", "VEILLE")
```

In [23]:
# Programme 16 — Démonstration de masques et np.where
conso = np.array([31.2, 148.5, 29.8, 152.1, 150.3, 0.0, 30.5])

# 1. Masque booléen
compresseur_actif = conso > 100.0
print("Masque booléen        :", compresseur_actif)
print("Valeurs actives       :", conso[compresseur_actif])
print("Nombre de points actifs:", np.sum(compresseur_actif))  # True compte pour 1, False pour 0

# 2. Remplacement conditionnel avec np.where :
# Si conso < 10 W (ex: capteur débranché ou 0), on remplace par NaN, sinon on garde la conso
conso_nettoyee = np.where(conso < 10.0, np.nan, conso)
print("Conso nettoyée        :", conso_nettoyee)

Masque booléen        : [False  True False  True  True False False]
Valeurs actives       : [148.5 152.1 150.3]
Nombre de points actifs: 3
Conso nettoyée        : [ 31.2 148.5  29.8 152.1 150.3   nan  30.5]


In [24]:
# Programme 17 — QCM 5
qcm(
    "Quelle est la syntaxe correcte pour sélectionner les éléments de 'a' compris strictement entre 10 et 50 ?",
    [
        "a[10 < a < 50]",
        "a[(a > 10) and (a < 50)]",
        "a[(a > 10) & (a < 50)]",
        "a[a > 10 , a < 50]"
    ],
    correct=2,
    explication="En NumPy, les conditions multiples nécessitent l'opérateur bitwise '&' et des parenthèses autour de chaque comparaison. L'opérateur Python 'and' évalue la vérité globale du tableau et génère une erreur."
)

### ✏️ Exercice 4 — Filtrage et binarisation d'état

Voici une série de 8 mesures de puissance d'une machine :
`mesures_p = np.array([25, 45, 130, 150, 142, 35, 160, 20])`

1. Isolez dans la variable `fortes_puissances` uniquement les valeurs strictement supérieures à **100 W**.
2. Créez un array `etat_machine` de même taille valant **1** si la puissance est supérieure ou égale à **40 W** (machine en fonction), et **0** sinon (utilisation recommandée de `np.where`).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
fortes_puissances = mesures_p[mesures_p > 100]
etat_machine = np.where(mesures_p >= 40, 1, 0)
```
</details>

In [ ]:
# Programme 18 — Exercice 4
mesures_p = np.array([25, 45, 130, 150, 142, 35, 160, 20])

# Remplacez les '...' par votre code :
fortes_puissances = ...
etat_machine = ...

In [ ]:
# Programme 19 — Test Exercice 4
check("fortes_puissances", fortes_puissances, np.array([130, 150, 142, 160]))
check("etat_machine", etat_machine, np.array([0, 1, 1, 1, 1, 0, 1, 0]))

---
# ⚡ 6 · Grand Cas Pratique : Audit de 99 396 Mesures Réelles de la Maison

Vous disposez maintenant des fondamentaux de NumPy à 1 dimension :
- la création d'arrays réguliers et le typage,
- les calculs vectorisés instantanés,
- le découpage temporel (*slicing*),
- et la puissance d'interrogation des **masques booléens** et de `np.where`.

Il est temps de mettre ces outils à l'épreuve sur des **données 100 % réelles** !

---

### 🏠 Le Jeu de Données Réel : Relevé Linky de la Maison
Ce jeu de données n'est pas une simulation théorique : il s'agit des mesures de puissance active (en Watts) issues du capteur de téléinformation Linky de la maison (projet `home-control`), enregistrées **toutes les 10 secondes** du **20 juin au 4 juillet 2026** (près de 14 jours continus).

NumPy possède son propre format de fichier binaire optimisé : le format **`.npy`**.  
Il permet de sauvegarder et charger des tableaux de millions de points de façon quasi instantanée (`np.load`).

Chargeons le fichier `data/conso_maison.npy` :

In [ ]:
# Programme 20 — Chargement des données réelles de la maison
conso_maison = np.load("data/conso_maison.npy")

print(f"✅ Relevé chargé dans 'conso_maison' : {conso_maison.size:,} mesures (dtype = {conso_maison.dtype})")
print(f"Premières valeurs (W) : {conso_maison[:8].tolist()} ...")
print(f"Puissance minimale    : {conso_maison.min()} W")
print(f"Puissance maximale    : {conso_maison.max()} W")

### ❓ Question 1 — Exploration du volume et de la durée couverte
À partir de l'array `conso_maison` :
1. Déterminez le nombre total de mesures `nb_mesures`.
2. Sachant qu'une mesure a lieu en moyenne toutes les 10 secondes, calculez la durée totale couverte en jours `duree_jours` (arrondie à 1 décimale).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_mesures = conso_maison.size
duree_jours = (nb_mesures * 10) / (24 * 3600)
```
</details>

In [ ]:
# Programme 21 — Cas Pratique : Question 1
# Remplacez les '...' par votre code :
nb_mesures = ...
duree_jours = ...

In [ ]:
# Programme 22 — Test Question 1
check("nb_mesures", nb_mesures, 99396)
check("duree_jours", duree_jours, 99396 * 10 / (24 * 3600), tol=0.1)

### ❓ Question 2 — Détection des pertes de trames (mesures à 0 W)
Dans un réseau de capteurs réels, il arrive que des paquets réseau soient perdus ou qu'une trame soit incomplète. Le récepteur enregistre alors `0 W` (valeur anormale pour une maison occupée qui a toujours au moins son talon de veille).

1. À l'aide d'un masque booléen, comptez combien de mesures sont égales à 0 dans `nb_mesures_nulles`.
2. Isolez dans la variable `conso_active` uniquement les mesures strictement positives ($P > 0$) grâce à l'indexation par masque.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_mesures_nulles = np.sum(conso_maison == 0)
conso_active = conso_maison[conso_maison > 0]
```
</details>

In [ ]:
# Programme 23 — Cas Pratique : Question 2
# Remplacez les '...' par votre code :
nb_mesures_nulles = ...
conso_active = ...

In [ ]:
# Programme 24 — Test Question 2
_ref_active = conso_maison[conso_maison > 0]

check("nb_mesures_nulles", nb_mesures_nulles, 12)
check("conso_active shape", conso_active.shape if hasattr(conso_active, 'shape') else None, _ref_active.shape)
check("conso_active min", conso_active.min() if hasattr(conso_active, 'min') else None, _ref_active.min())

### ❓ Question 3 — Audit du disjoncteur (Abonnement 6 kVA)
La maison dispose d'un contrat d'électricité standard de **6 kVA** (soit une puissance maximale autorisée de **6 000 Watts**).
1. Trouvez la puissance maximale absolue enregistrée sur l'ensemble du relevé dans `puissance_max`.
2. À l'aide d'un masque booléen, comptez combien de points ont dépassé le seuil de 6 000 W dans `nb_points_depassement`.
3. Sachant que 1 point = 10 secondes, calculez la durée totale cumulée de dépassement en minutes dans `duree_depassement_min`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
puissance_max = conso_maison.max()
nb_points_depassement = np.sum(conso_maison > 6000)
duree_depassement_min = (nb_points_depassement * 10) / 60
```
</details>

In [ ]:
# Programme 25 — Cas Pratique : Question 3
# Remplacez les '...' par votre code :
puissance_max = ...
nb_points_depassement = ...
duree_depassement_min = ...

In [ ]:
# Programme 26 — Test Question 3
check("puissance_max", puissance_max, 7300.0)
check("nb_points_depassement", nb_points_depassement, 51)
check("duree_depassement_min", duree_depassement_min, 51 * 10 / 60, tol=0.1)

### ❓ Question 4 — Identification des appareils énergivores (> 2 000 W)
Un appareil qui consomme plus de 2 000 W (four, chauffe-eau, plaques, lave-linge) pèse lourdement sur la consommation.
1. Calculez le nombre d'heures totales cumulées où la puissance a dépassé 2 000 W dans `heures_gros_appareils` (rappel : 1 heure = 360 mesures).
2. Calculez quel pourcentage du temps global cela représente dans `pourcentage_temps_fort`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
heures_gros_appareils = np.sum(conso_maison > 2000) * 10 / 3600
pourcentage_temps_fort = (np.sum(conso_maison > 2000) / conso_maison.size) * 100
```
</details>

In [ ]:
# Programme 27 — Cas Pratique : Question 4
# Remplacez les '...' par votre code :
heures_gros_appareils = ...
pourcentage_temps_fort = ...

In [ ]:
# Programme 28 — Test Question 4
_heures_ref = float(np.sum(conso_maison > 2000) * 10 / 3600)
_pct_ref = float((np.sum(conso_maison > 2000) / conso_maison.size) * 100)

check("heures_gros_appareils", heures_gros_appareils, _heures_ref, tol=0.05)
check("pourcentage_temps_fort", pourcentage_temps_fort, _pct_ref, tol=0.05)

### ❓ Question 5 — Bilan d'énergie consommée et estimation de la facture
La puissance $P$ s'exprime en Watts (débit instantané). L'énergie $E$ s'exprime en kiloWatt-heures (kWh) :
$$E = \frac{\sum (P_i \times \Delta t)}{1000 \times 3600}$$
avec $\Delta t = 10\text{ s}$.

1. Calculez l'énergie totale consommée par la maison sur l'ensemble de la période dans `energie_totale_kwh`.
2. Au tarif de **0,25 € par kWh**, calculez le montant estimé de la facture d'électricité dans `facture_estimee_eur`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
energie_totale_kwh = np.sum(conso_maison * 10) / (1000 * 3600)
facture_estimee_eur = energie_totale_kwh * 0.25
```
</details>

In [ ]:
# Programme 29 — Cas Pratique : Question 5
# Remplacez les '...' par votre code :
energie_totale_kwh = ...
facture_estimee_eur = ...

In [ ]:
# Programme 30 — Test Question 5
_energie_ref = float(np.sum(conso_maison * 10) / (1000 * 3600))
check("energie_totale_kwh", energie_totale_kwh, _energie_ref, tol=0.2)
check("facture_estimee_eur", facture_estimee_eur, _energie_ref * 0.25, tol=0.1)

### ❓ Question 6 — Zoom par Slicing : Les premières 24 heures
Grâce au slicing de NumPy, on peut découper des tranches temporelles précises sans boucle.  
Puisque 1 heure = 360 mesures, les 24 premières heures correspondent aux **8 640 premières mesures** ($24 \times 360$).

1. Extrayez ce premier jour complet dans `jour_1` sous forme d'une copie indépendante avec `.copy()`.
2. Calculez sa puissance moyenne dans `puissance_moyenne_j1`.
3. Trouvez la puissance médiane de cette première journée dans `mediane_j1` avec `np.median()`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
jour_1 = conso_maison[: 24 * 360].copy()
puissance_moyenne_j1 = jour_1.mean()
mediane_j1 = np.median(jour_1)
```
</details>

In [ ]:
# Programme 31 — Cas Pratique : Question 6
# Remplacez les '...' par votre code :
jour_1 = ...
puissance_moyenne_j1 = ...
mediane_j1 = ...

In [ ]:
# Programme 32 — Test Question 6
_j1_ref = conso_maison[: 24 * 360]

check("jour_1 shape", jour_1.shape if hasattr(jour_1, 'shape') else None, (8640,))
check("puissance_moyenne_j1", puissance_moyenne_j1, float(_j1_ref.mean()), tol=0.5)
check("mediane_j1", mediane_j1, float(np.median(_j1_ref)), tol=0.5)

### 👁️ Pour aller plus loin : Visualisation de confirmation
Vous venez de mener un audit quantitatif complet sur près de **100 000 mesures réelles** uniquement avec NumPy, sans tracer une seule courbe !

Pour contempler le profil que vous venez d'analyser, exécutez la cellule ci-dessous : elle trace les **24 premières heures** réelles de la maison avec le seuil des gros appareils et le talon de veille.

In [33]:
# Programme 33 — Visualisation des premières 24 heures réelles
points_24h = 24 * 360  # 8 640 points
t_heures_24 = np.arange(points_24h) * 10 / 3600.0

plt.figure(figsize=(14, 5))
plt.plot(t_heures_24, conso_maison[:points_24h], color='#0d6efd', linewidth=0.9, label='Relevé réel (Watts)')
plt.axhline(2000, color='#dc3545', linestyle='--', linewidth=1.5, label='Seuil gros appareils (2 000 W)')
plt.axhline(750, color='#198754', linestyle=':', linewidth=1.5, label='Médiane de fond (~750 W)')

plt.title("🏠 Relevé Réel Linky — Les premières 24 heures de la maison", fontsize=13, fontweight='bold')
plt.xlabel("Heures écoulées (0h à 24h)", fontsize=11)
plt.ylabel("Puissance active (Watts)", fontsize=11)
plt.xlim(0, 24)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

/var/folders/kz/lyx_lm0532g9q8gdt228qdmw0000gn/T/ipykernel_43840/3819256020.py:16: UserWarning: Glyph 127968 (\N{HOUSE BUILDING}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/kz/lyx_lm0532g9q8gdt228qdmw0000gn/T/ipykernel_43840/3819256020.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7 · Synthèse de Tool 1 & Prochaine Étape

Félicitations ! Vous avez validé les fondations du calcul vectoriel à 1 dimension sur des données réelles :
- La différence fondamentale entre listes et arrays contigus en mémoire.
- La syntaxe vectorisée sans boucle.
- La gestion rigoureuse de la mémoire (`.copy()`).
- Le filtrage par masque booléen (`&`, `|`, `~`) et le remplacement avec `np.where`.
- L'audit quantitatif sur près de 100 000 mesures réelles.

---

### 🚀 Rendez-vous dans `tool2.ipynb` !
Dans le notebook suivant, nous passerons aux dimensions supérieures avec :
1. **Les tableaux à 2 dimensions (matrices)** et la réorganisation de forme (`reshape`, `newaxis`).
2. **Le paramètre `axis`** (`axis=0` vs `axis=1`) pour agréger par ligne ou par colonne.
3. **Le Broadcasting**, la technique secrète pour opérer entre matrices et vecteurs.
4. **La génération aléatoire moderne** (`np.random.default_rng`).
5. **Le lissage de signal par convolution** (`np.convolve`).